# Text Embeddings using Universal Sentence Encoder

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow.keras as keras

In [4]:
model_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
model = hub.load(model_url)

In [5]:
def embed(input_text, embed_model=model):
    return embed_model(input_text)

In [6]:
embeddings = embed(['This is a sentence'])

In [7]:
embeddings

<tf.Tensor: shape=(1, 512), dtype=float32, numpy=
array([[ 0.02881767, -0.02020015,  0.01069628,  0.0385053 , -0.09253702,
         0.01752774, -0.04711751,  0.0478521 ,  0.01430713,  0.02635952,
         0.02157276, -0.04987606,  0.03023818,  0.11923701,  0.04333102,
        -0.11343763, -0.02569381,  0.00875983, -0.03205026, -0.03528893,
         0.065098  ,  0.07168703,  0.00487422, -0.00622953, -0.05483385,
         0.06636366,  0.02197697, -0.08987362,  0.03663914, -0.04186109,
         0.03020706, -0.02176558,  0.00683554, -0.04167067, -0.09498136,
        -0.03249762,  0.06278758,  0.04457697,  0.00825471, -0.02461435,
        -0.00744968,  0.00276056,  0.04595318,  0.0438056 , -0.06370507,
        -0.01170835, -0.05826195, -0.02410091, -0.035766  , -0.03617879,
        -0.01113707, -0.0866305 , -0.08743551,  0.00281444, -0.01570992,
        -0.04420159,  0.0287745 ,  0.03433856, -0.0100031 , -0.04693236,
         0.00529842, -0.05397878,  0.01005255, -0.01533142, -0.05143673,
 

## Semenatic Similarity Scoring

In [8]:
from numpy.linalg import norm

In [9]:
def cos_sim(A, B):
    return np.inner(A,B)/(norm(A)*norm(B))

def euclidean(A,B):
    return norm(A-B)

In [10]:
def is_it_sim(textA, textB, thresh=.25, sim_func=cos_sim):
    embeddingsA = embed([textA])
    embeddingsB = embed([textB])
    sim_score = sim_func(embeddingsA, embeddingsB)
    if sim_score > thresh:
        print("They are SIMILAR!")
    else:
        print("Not at all...")
    return sim_score, sim_score>thresh

In [11]:
QuestionA = 'This is a technology that builds computers'

AnswerA = ' I had a delicious fruit called an apple'
AnswerB = 'I am writing a program on my Apple laptop'

is_it_sim(QuestionA, AnswerA)

Not at all...


(array([[0.07709729]], dtype=float32), array([[False]]))

In [12]:
is_it_sim(QuestionA, AnswerB)

They are SIMILAR!


(array([[0.26977792]], dtype=float32), array([[ True]]))

# Text Classification using Embeddings and GUSE

In [1]:
!pip install tf_keras
!pip install tensorflow_datasets

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_hub as hub
import tensorflow.keras as keras
import tf_keras

import tensorflow_datasets as tfds

In [3]:
train_data, validation_data, test_data = tfds.load(
    name="imdb_reviews",
    split=('train[:60%]','train[60%:]','test'),
    as_supervised =True
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.FP8JAM_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.FP8JAM_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.FP8JAM_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [4]:
train_examples_batch, train_labels_batch = next(iter(train_data.batch(10)))

## Load Hub Layer and Create a Hub Layer

In [5]:
embedding_model_url = 'https://tfhub.dev/google/universal-sentence-encoder/4'

In [6]:
hub_layer = hub.KerasLayer(embedding_model_url, input_shape=[], dtype=tf.string, trainable=False)

## Build a Neural Net Text Classifier

In [7]:
nlp_model = tf_keras.Sequential()

In [8]:
nlp_model.add(hub_layer)
nlp_model.add(tf_keras.layers.Dense(256, activation='relu'))
nlp_model.add(tf_keras.layers.Dropout(.1))
nlp_model.add(tf_keras.layers.Dense(128, activation='relu'))
nlp_model.add(tf_keras.layers.Dropout(.1))
nlp_model.add(tf_keras.layers.Dense(64, activation='relu'))
nlp_model.add(tf_keras.layers.Dense(1, activation='sigmoid'))

In [9]:
nlp_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 512)               256797824 
                                                                 
 dense (Dense)               (None, 256)               131328    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)             (None, 64)                8256      
                                                                 
 dense_3 (Dense)             (None, 1)                 6

## Compile and Train NLP Model

In [10]:
nlp_model.compile(optimizer='Adam', loss=tf_keras.losses.BinaryCrossentropy(from_logits=False), metrics=['binary_accuracy'])

In [12]:
train_data.batch(512)

<_BatchDataset element_spec=(TensorSpec(shape=(None,), dtype=tf.string, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [13]:
history = nlp_model.fit(train_data.batch(512), epochs=20, validation_data=validation_data.batch(512), verbose=2)

Epoch 1/20
30/30 - 41s - loss: 0.2529 - binary_accuracy: 0.8991 - val_loss: 0.3326 - val_binary_accuracy: 0.8634 - 41s/epoch - 1s/step
Epoch 2/20
30/30 - 28s - loss: 0.2282 - binary_accuracy: 0.9132 - val_loss: 0.3451 - val_binary_accuracy: 0.8586 - 28s/epoch - 930ms/step
Epoch 3/20
30/30 - 27s - loss: 0.2070 - binary_accuracy: 0.9202 - val_loss: 0.3525 - val_binary_accuracy: 0.8582 - 27s/epoch - 900ms/step
Epoch 4/20
30/30 - 27s - loss: 0.1863 - binary_accuracy: 0.9299 - val_loss: 0.3795 - val_binary_accuracy: 0.8515 - 27s/epoch - 902ms/step
Epoch 5/20
30/30 - 28s - loss: 0.1702 - binary_accuracy: 0.9376 - val_loss: 0.3734 - val_binary_accuracy: 0.8602 - 28s/epoch - 918ms/step
Epoch 6/20


KeyboardInterrupt: 

## Model Evaluation

In [14]:
results = nlp_model.evaluate(test_data.batch(512), verbose=2)

49/49 - 27s - loss: 0.4321 - binary_accuracy: 0.8356 - 27s/epoch - 560ms/step


## Model Save

In [15]:
import os
if not os.path.exists('/content/TFModels/'):
  os.mkdir('/content/TFModels/')

In [16]:
nlp_model.save('/content/TFModels')

## Inference Function

In [21]:
def get_sentiment(text, model=nlp_model, thresh=0.5):
  p_hat =model.predict([text])[0][0]
  out = (p_hat>thresh).astype('int32')

  print("Viewer Comment:\n"+text+"\n\n The Review was:")
  if out: print(" It was Good!")
  else: print("Bad Movie!")
  return p_hat


In [26]:
get_sentiment('It was Dynamite! Will see again, and again, if I wanted to barf a lot!')

1/1 [==============================] - 0s 36ms/step
Viewer Comment:
It was Dynamite! Will see again, and again, if I wanted to barf a lot!

 The Review was:
Bad Movie!


np.float32(0.0078052534)

In [29]:
get_sentiment("Few times in my life a movie makes me think about the truth and beauty in the world")

1/1 [==============================] - 0s 72ms/step
Viewer Comment:
Few times in my life a movie makes me think about the truth and beauty in the world

 The Review was:
 It was Good!


np.float32(0.9924534)